In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
ods_dir = Path("/data-ssd/alex/datasets/inat-open-data/inaturalist-open-data-20240827/")
export_dir = Path("/data-ssd/alex/datasets/vision-export-20230412214019-Mammalia-40151/")

In [3]:
ods_obs = pd.read_csv(ods_dir / "obs_photos_mammals.csv")
ods_obs.sample(3)

,observation_uuid,taxon_id,photo_id,extension
1712904,00af0477-aa61-4fdc-b3af-ffbd32a504c4,204114,153110080,jpg
3792551,20bbfc5a-915a-405e-b50f-493711d2eab4,41979,347915905,jpeg
235161,87ef06d9-2de2-49ef-bc40-b8b4dc5f1a60,44026,14448339,jpeg


In [5]:
val_data = pd.read_csv(
    export_dir / "val_cleaned.csv", 
    usecols=["filename", "photo_id", "taxon_id", "leaf_class_id"]
)
train_data = pd.read_csv(
    export_dir / "train_cleaned.csv",
    usecols=["filename", "photo_id", "taxon_id", "leaf_class_id"]
)
tax = pd.read_csv(
    export_dir / "taxonomy.csv",
)

In [7]:
val_photo_ids = set(val_data.photo_id.values)
train_photo_ids = set(train_data.photo_id.values)

In [9]:
ods_obs_new = ods_obs.copy()
ods_obs_new = ods_obs_new[~ ods_obs_new.photo_id.isin(val_photo_ids) ]
ods_obs_new = ods_obs_new[~ ods_obs_new.photo_id.isin(train_photo_ids) ]

In [10]:
ods_obs_new.taxon_id = ods_obs_new.taxon_id.astype(np.int32)

In [11]:
ods_obs_new = ods_obs_new[["photo_id", "extension", "taxon_id"]]

In [12]:
ods_obs_new = pd.merge(
    ods_obs_new,
    tax,
    left_on="taxon_id",
    right_on="taxon_id",
)
ods_obs_new.leaf_class_id = ods_obs_new.leaf_class_id.astype(np.int32)
ods_obs_new = ods_obs_new[["photo_id", "extension", "taxon_id", "leaf_class_id"]]

In [13]:
ods_obs_new.to_csv(
    export_dir / "train_data_augmented.csv",
    index=False,
)